# Consumer Complaint Classification - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the consumer complaint dataset.

## Steps:
1. Load and inspect the data
2. Analyze class distribution
3. Text statistics and analysis
4. Word clouds for each category
5. N-gram analysis
6. Identify patterns and insights

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

%matplotlib inline

## 1. Load Data

In [ ]:
from src.data_loader import load_data, map_categories

# Load data (adjust path as needed)
# NOTE: Download the dataset from https://catalog.data.gov/dataset/consumer-complaint-database
df = load_data(
    'data/raw/complaints.csv',
    sample_size=50000  # Use sample for faster analysis
)

# Map categories
df = map_categories(df)

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Class Distribution Analysis

In [ ]:
# Category mapping
category_names = {
    0: "Credit reporting, repair, or other",
    1: "Debt collection",
    2: "Consumer Loan",
    3: "Mortgage"
}

# Count by category
category_counts = df['label_encoded'].value_counts().sort_index()

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
ax1 = axes[0]
category_counts.plot(kind='bar', ax=ax1, color='skyblue', edgecolor='black')
ax1.set_title('Complaint Distribution by Category', fontsize=14, fontweight='bold')
ax1.set_xlabel('Category', fontsize=12)
ax1.set_ylabel('Number of Complaints', fontsize=12)
ax1.set_xticklabels([category_names[i] for i in category_counts.index], rotation=45, ha='right')
ax1.grid(axis='y', alpha=0.3)

# Pie chart
ax2 = axes[1]
colors = plt.cm.Set3(np.linspace(0, 1, len(category_counts)))
ax2.pie(category_counts.values, labels=[category_names[i] for i in category_counts.index],
        autopct='%1.1f%%', startangle=90, colors=colors)
ax2.set_title('Category Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/category_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics
print("\nCategory Statistics:")
for idx, count in category_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{category_names[idx]:40s}: {count:6d} ({percentage:5.2f}%)")

## 3. Text Statistics Analysis

In [ ]:
from src.preprocessing import get_text_statistics

# Calculate text statistics
df_stats = get_text_statistics(df, 'text')

# Summary statistics
print("Text Statistics Summary:")
print(df_stats[['char_count', 'word_count', 'avg_word_length']].describe())

# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Character count distribution
axes[0, 0].hist(df_stats['char_count'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Character Count Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Number of Characters')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df_stats['char_count'].median(), color='red', linestyle='--', label=f'Median: {df_stats["char_count"].median():.0f}')
axes[0, 0].legend()

# Word count distribution
axes[0, 1].hist(df_stats['word_count'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Word Count Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Number of Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df_stats['word_count'].median(), color='red', linestyle='--', label=f'Median: {df_stats["word_count"].median():.0f}')
axes[0, 1].legend()

# Average word length by category
df_stats['category'] = df_stats['label_encoded'].map(category_names)
df_stats.boxplot(column='avg_word_length', by='category', ax=axes[1, 0])
axes[1, 0].set_title('Average Word Length by Category', fontweight='bold')
axes[1, 0].set_xlabel('Category')
axes[1, 0].set_ylabel('Average Word Length')
plt.sca(axes[1, 0])
plt.xticks(rotation=45, ha='right')

# Word count by category
df_stats.boxplot(column='word_count', by='category', ax=axes[1, 1])
axes[1, 1].set_title('Word Count by Category', fontweight='bold')
axes[1, 1].set_xlabel('Category')
axes[1, 1].set_ylabel('Word Count')
axes[1, 1].set_ylim(0, 500)  # Limit for better visibility
plt.sca(axes[1, 1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../results/figures/text_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Word Clouds by Category

In [ ]:
from src.preprocessing import TextPreprocessor

# Initialize preprocessor
preprocessor = TextPreprocessor(
    lowercase=True,
    remove_urls=True,
    remove_emails=True,
    remove_stopwords=True,
    lemmatize=True
)

# Preprocess texts
df['processed_text'] = df['text'].apply(preprocessor.preprocess)

# Generate word clouds for each category
fig, axes = plt.subplots(2, 2, figsize=(20, 15))
axes = axes.ravel()

for idx, (cat_id, cat_name) in enumerate(category_names.items()):
    # Get text for this category
    cat_text = ' '.join(df[df['label_encoded'] == cat_id]['processed_text'].values)
    
    # Generate word cloud
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        max_words=100
    ).generate(cat_text)
    
    # Plot
    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].set_title(f'Word Cloud - {cat_name}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/wordclouds.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. N-gram Analysis

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(corpus, n=2, top_k=20):
    """Get top k n-grams from corpus"""
    vectorizer = CountVectorizer(ngram_range=(n, n), max_features=top_k)
    X = vectorizer.fit_transform(corpus)
    
    ngrams = vectorizer.get_feature_names_out()
    counts = X.sum(axis=0).A1
    
    ngram_freq = dict(zip(ngrams, counts))
    return sorted(ngram_freq.items(), key=lambda x: x[1], reverse=True)

# Analyze bigrams for each category
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.ravel()

for idx, (cat_id, cat_name) in enumerate(category_names.items()):
    cat_texts = df[df['label_encoded'] == cat_id]['processed_text'].values
    
    # Get top bigrams
    top_bigrams = get_top_ngrams(cat_texts, n=2, top_k=15)
    
    # Plot
    bigrams, counts = zip(*top_bigrams)
    y_pos = np.arange(len(bigrams))
    
    axes[idx].barh(y_pos, counts, color='teal', edgecolor='black')
    axes[idx].set_yticks(y_pos)
    axes[idx].set_yticklabels(bigrams)
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Frequency', fontweight='bold')
    axes[idx].set_title(f'Top Bigrams - {cat_name}', fontsize=12, fontweight='bold')
    axes[idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/bigrams_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
# Analyze correlation between text features
feature_cols = ['char_count', 'word_count', 'avg_word_length', 'stopword_count']
correlation_matrix = df_stats[feature_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Sample Complaints by Category

In [ ]:
# Display sample complaints from each category
print("Sample Complaints by Category:\n")
print("=" * 80)

for cat_id, cat_name in category_names.items():
    print(f"\n{cat_name.upper()}")
    print("-" * 80)
    
    samples = df[df['label_encoded'] == cat_id]['text'].sample(min(2, len(df[df['label_encoded'] == cat_id])))
    
    for i, text in enumerate(samples.values, 1):
        print(f"\nSample {i}:")
        print(text[:300] + "..." if len(text) > 300 else text)
    
    print("-" * 80)

## Summary and Insights

Based on the EDA, we can observe:

1. **Class Distribution**: Note any class imbalance
2. **Text Length**: Typical complaint length and variation
3. **Key Terms**: Important words and phrases for each category
4. **Patterns**: Common complaint patterns

These insights will guide:
- Preprocessing strategies
- Feature engineering approaches
- Model selection
- Handling class imbalance